## Baixando a base de dados diretamente do DATASUS

In [35]:
import pandas as pd
import os
import re
import unicodedata

### Código para verificar o que tem nas pastas

In [ ]:
from ftplib import FTP

ftp = FTP('ftp.datasus.gov.br')
ftp.login()
ftp.cwd('/dissemin/publicos/SISCAN/SISCAN')

# Vamos ver o que tem dentro da pasta SISCAN
print("Conteúdo de /SISCAN/ :", ftp.nlst())

# Se aparecer 'CO' ou 'MA', vamos entrar em uma delas para ver a estrutura
# Exemplo para Colo de Útero:
try:
    ftp.cwd('CO')
    print("Conteúdo de /SISCAN/CO/ :", ftp.nlst())
except:
    print("Não existe subpasta CO.")

ftp.quit()

In [ ]:
def baixar_csv_siscan(arquivos_desejados):
    ftp = FTP('ftp.datasus.gov.br')
    ftp.login()
    ftp.cwd('/dissemin/publicos/SISCAN/SISCAN')
    
    for nome_arq in arquivos_desejados:
        print(f"Baixando {nome_arq}...")
        try:
            with open(nome_arq, 'wb') as f:
                ftp.retrbinary(f'RETR {nome_arq}', f.write)
            print(f"Sucesso: {nome_arq}")
        except Exception as e:
            print(f"Erro ao baixar {nome_arq}: {e}")
            
    ftp.quit()

# Lista baseada no que você encontrou (ajuste conforme seu interesse: COLO ou MAMA)
meus_arquivos = [
    # 'SISCAN_CITO_COLO_2025.csv',
    'SISCAN_HISTO_COLO_2025.csv',
    'SISCAN_HISTO_COLO_PACNT_2025.csv',
    'SISCAN_CITO_COLO_PACNT_2025.csv'
    # 'TAB_SISCAN.rar'
]

baixar_csv_siscan(meus_arquivos)

In [ ]:
cols_cito_interesses = ['CO_IDADE_PACIENTE', 'CO_RACA_COR', 'CO_ESCOLARIDADE', 'CO_UF_RESIDENCIA', 'NU_EXAME_ALTERADO', 'NU_LESAO_IE_ALTO_GRAU', 'NU_LESAO_IE_BAIXO_GRAU', 'NU_CARC_EPIDERM_INV', 'ST_DENTRO_LIMITE_NORMALIDADE', 'ST_SINAL_SUGESTIVO_DST', 'TP_ATIPIA_CELULAR_ESCAMOS', 'CO_MIC_CANDIDA', 'CO_MIC_TRICHOMONAS','CO_LAUDO_CITOPATOLOGICO', 'TP_MOTIVO_EXAME', 'ST_REPRESENTATI_ZONA_TRANSFORM','TP_ATIPIA_CELULAR_GLANDUL']

cols_histo_interesses = ['NU_NEOPLASICO_NIC_I', 'NU_NEOPLASICO_NIC_II', 'NU_NEOPLASICO_NIC_III', 'NU_NEO_CARC_EPID_INVASIVO', 'NU_LES_BEN_CERVICITE_CRON', 'NU_RES_EXAMES_NEO_PRENEO', 'TP_ACHADO_COLPOSCOPICO', 'TP_GRAU_DIFERENCIACAO', 'TP_PROCEDIMENTO', 'TP_ENCAMINHAMENTO', 'TP_MARGEM_CIRURGICA']

In [18]:
pd.set_option('display.max_columns', None)
# Para ler no Pandas depois de baixar:
df_cito = pd.read_csv('SISCAN_CITO_COLO_2025.csv', usecols=cols_cito_interesses,sep=';',           # O segredo costuma estar aqui
        encoding='latin1')
df_histo = pd.read_csv('SISCAN_HISTO_COLO_2025.csv', usecols=cols_histo_interesses,sep=';',           # O segredo costuma estar aqui
        encoding='latin1')

### Próximos passos:
    - [X] Ler o dicionário baixado que explica as colunas "Tab_siscan.rar"
    - [] Buscar criar um dicionário das colunas

In [25]:
def carregar_todos_cnvs(pasta_cnv):
    mapa_mestre = {}
    # Lista todos os arquivos na pasta onde você descompactou o .rar
    for arquivo in os.listdir(pasta_cnv):
        if arquivo.endswith('.cnv'):
            # O nome do dicionário será o nome do arquivo (ex: RACA)
            nome_dicio = arquivo.replace('.cnv', '').upper()
            traducao_simples = {}
            
            # Abre o arquivo .cnv (sempre use iso-8859-1 para DATASUS)
            with open(os.path.join(pasta_cnv, arquivo), 'r', encoding='iso-8859-1') as f:
                for linha in f:
                    partes = linha.split()
                    if len(partes) >= 2:
                        codigo = partes[0]
                        descricao = " ".join(partes[1:])
                        traducao_simples[codigo] = descricao
            
            mapa_mestre[nome_dicio] = traducao_simples
    return mapa_mestre

# Uso:
dicionarios = carregar_todos_cnvs('./cnvs')

In [34]:
dicionarios

{'DIAGN_IMAGEM': {'4': 'Assimetria 04',
  '1': 'Microcalcificação 01',
  '2': 'Distorção 02',
  '3': 'Nódulo 03'},
 'ANO': {'10': '2022 2022',
  '1': '2013 2013',
  '2': '2014 2014',
  '3': '2015 2015',
  '4': '2016 2016',
  '5': '2017 2017',
  '6': '2018 2018',
  '7': '2019 2019',
  '8': '2020 2020',
  '9': '2021 2021'},
 'OUTR_IMUNO_HISTOQ': {'2': 'Não N', '1': 'Sim S'},
 'TWUSAUDE': {'474': '9101039 USF MANOEL BATISTA 9101039',
  '1': '2266725 POSTO DE SAUDE PONTA NEGRA 2266725',
  '2': '2266768 PSF BARRA 2266768',
  '3': '2266784 PSF JARDIM ATLANTICO 2266784',
  '4': '2266792 POSTO DE SAUDE SANTA RITA 2266792',
  '5': '2266806 PSF PONTA GROSSA 2266806',
  '6': '2266822 PSF RECANTO 2266822',
  '7': '2266849 PSF DO ESPRAIADO 2266849',
  '8': '2266857 POSTO DE SAUDE SAO JOSE 2266857',
  '9': '2266865 PSF BAMBUI 2266865',
  '10': '2266873 POSTO DE SAUDE INOA 2266873',
  '11': '2266881 POSTO DE SAUDE CENTRAL 2266881',
  '12': '2266911 PSF BAIRRO DA AMIZADE 2266911',
  '13': '2266938 PSF

In [36]:
def normalizar_nome(texto):
    if not isinstance(texto, str):
        return texto
    # 1. Remover acentos e converter para minúsculas
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8').lower()
    # 2. Remover prefixos comuns do DATASUS (CO_, TP_, SG_, NU_)
    texto = re.sub(r'^(co_|tp_|sg_|nu_|st_)', '', texto)
    # 3. Remover caracteres especiais e espaços (mantendo apenas letras e números)
    texto = re.sub(r'[^a-z0-9]', '', texto)
    return texto

In [38]:
def gerar_documentacao_e_traduzir(df, dicionarios_originais):
    # Criamos versões "limpas" das chaves dos dicionários para busca
    dict_limpo = {normalizar_nome(k): k for k in dicionarios_originais.keys()}
    
    df_traduzido = df.copy()
    linhas_doc = [] # Para o dicionario.md

    for col in df.columns:
        col_norm = normalizar_nome(col)
        encontrou = False
        
        # Busca por correspondência exata ou parcial no dicionário limpo
        for chave_norm, chave_original in dict_limpo.items():
            # Se o nome da coluna limpa está contido na chave do CNV ou vice-versa
            if chave_norm in col_norm or col_norm in chave_norm:
                dict_valores = dicionarios_originais[chave_original]
                
                # 1. Traduz o DataFrame (Cria coluna _DESC)
                df_traduzido[f"{col}_DESC"] = df_traduzido[col].astype(str).map(dict_valores)
                
                # 2. Registra para o dicionario.md
                # Pegamos os primeiros 3 valores como exemplo
                exemplos = list(dict_valores.values())[:3]
                valores_str = ", ".join(exemplos) + "..."
                
                linhas_doc.append({
                    "Coluna": col,
                    "Explicação": f"Mapeado via {chave_original}",
                    "Valores/Significados": valores_str
                })
                encontrou = True
                break
        
        if not encontrou:
            linhas_doc.append({
                "Coluna": col,
                "Explicação": "Dado bruto / Sem dicionário encontrado",
                "Valores/Significados": "Numérico ou ID"
            })

    # Criar o Markdown
    df_doc = pd.DataFrame(linhas_doc)
    markdown_table = df_doc.to_markdown(index=False)
    
    return df_traduzido, markdown_table

# Execução
df_final, tabela_md = gerar_documentacao_e_traduzir(df_cito, dicionarios)

# Salvar o arquivo
with open("dicionario.md", "w", encoding="utf-8") as f:
    f.write("# Dicionário de Dados Automatizado\n\n")
    f.write(tabela_md)

ImportError: Missing optional dependency 'tabulate'.  Use pip or conda to install tabulate.

In [32]:
# Uso:
df_cito_final = aplicar_dicionarios_automatico(df_cito, dicionarios)

CO_UF_RESIDENCIA UF_RESIDENCIA
Chave: DIAGN_IMAGEM
Chave: ANO
Chave: OUTR_IMUNO_HISTOQ
Chave: TWUSAUDE
Chave: BR_UFALFA
Chave: ETNIA
Chave: TP_RECEPTOR
Chave: BR_REGSAUD
Chave: ADEQUIB_ZT
Chave: BR_MUNICIP
Chave: TPENCAMINH
Chave: TPPROCED
Chave: BR_REGIAO
Chave: TIPMAMRAST
Chave: BR_UF
Chave: LAUDO_CITOPATOLOGICO
Chave: TPPROCCIR
CO_RACA_COR RACA_COR
Chave: DIAGN_IMAGEM
Chave: ANO
Chave: OUTR_IMUNO_HISTOQ
Chave: TWUSAUDE
Chave: BR_UFALFA
Chave: ETNIA
Chave: TP_RECEPTOR
Chave: BR_REGSAUD
Chave: ADEQUIB_ZT
Chave: BR_MUNICIP
Chave: TPENCAMINH
Chave: TPPROCED
Chave: BR_REGIAO
Chave: TIPMAMRAST
Chave: BR_UF
Chave: LAUDO_CITOPATOLOGICO
Chave: TPPROCCIR
CO_IDADE_PACIENTE IDADE_PACIENTE
Chave: DIAGN_IMAGEM
Chave: ANO
Chave: OUTR_IMUNO_HISTOQ
Chave: TWUSAUDE
Chave: BR_UFALFA
Chave: ETNIA
Chave: TP_RECEPTOR
Chave: BR_REGSAUD
Chave: ADEQUIB_ZT
Chave: BR_MUNICIP
Chave: TPENCAMINH
Chave: TPPROCED
Chave: BR_REGIAO
Chave: TIPMAMRAST
Chave: BR_UF
Chave: LAUDO_CITOPATOLOGICO
Chave: TPPROCCIR
CO_ESCOLA

In [33]:
df_cito_final

,CO_UF_RESIDENCIA,CO_RACA_COR,CO_IDADE_PACIENTE,CO_ESCOLARIDADE,ST_SINAL_SUGESTIVO_DST,ST_DENTRO_LIMITE_NORMALIDADE,TP_ATIPIA_CELULAR_ESCAMOS,CO_MIC_CANDIDA,CO_MIC_TRICHOMONAS,NU_LESAO_IE_BAIXO_GRAU,NU_LESAO_IE_ALTO_GRAU,NU_CARC_EPIDERM_INV,NU_EXAME_ALTERADO
0,17,3,34,NaN,N,N,NaN,0.0,0.0,0,0,0,0
1,15,3,21,NaN,N,N,NaN,0.0,0.0,0,0,0,0
2,15,4,35,NaN,N,N,NaN,0.0,0.0,0,0,0,0
3,15,3,50,NaN,N,N,NaN,0.0,0.0,0,0,0,0
4,15,1,45,NaN,N,N,NaN,0.0,0.0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6851033,26,3,37,NaN,N,N,NaN,0.0,0.0,0,0,0,0
6851034,31,3,55,NaN,N,N,NaN,0.0,0.0,0,0,0,0
6851035,42,1,26,NaN,N,N,NaN,0.0,0.0,0,0,0,0
6851036,31,2,37,NaN,N,N,NaN,0.0,0.0,0,0,0,0
